
# 03L — Corrected Local Fireline Unit Production

**Build ID: `03L_CORRECTED_LOCAL_UNIT_PRODUCTION_V1`**

03L implements the corrected production representation established by the fragmentation audits.

Fixed geometry settings:

\[
\boxed{L_{\max}=250\text{ m},\qquad g_{\max}=10\text{ m}}
\]

Fixed temporal settings:

\[
\boxed{25\text{ m raster},\qquad 50\text{ m challenge tolerance},\qquad 48\text{ h maximum NIROPS interval}}
\]

Long continuous components keep the accepted 03G 250 m segmentation boundaries. Sub-250 m components may be grouped only within the **same exact parent geometry hash** and only through endpoint gaps ≤10 m. No artificial bridge line is inserted.

Each corrected unit preserves its actual geometry. The geometry is sampled at approximately 25 m spacing for NIROPS reconstruction, with every constituent geometry part receiving at least one sample location.

### Performance design

- Canonical parent geometries are sharded once and reused.
- Geometry explosion is vectorized with GeoPandas rather than looping over every parent.
- Singleton short components bypass KD-tree graph construction.
- Only the frozen 10 m graph is built.
- LineString sampling is vectorized in Shapely for the common case.
- NIROPS is rasterized once per observation/region and sampled for every point at once.
- Signed trajectories are cached per incident.
- All expensive stages are resumable.
- Incidents are processed smallest-first.
- Processing is sequential by incident to avoid distance-transform memory contention.

### Candidate unit-level temporal rule

**Burned Over:** supported if at least one sample point shows a supported outside→inside transition.

**Held:** supported if at least one sample point shows supported outside→near engagement **and no sample point is a Held trajectory conflict**.

The notebook also stores any/majority/all-supported fractions so the candidate rule can be audited before the corrected cohort is frozen.


In [ ]:
import os  # Configurable project root for the public release.

NOTEBOOK_BUILD_ID = "03L_CORRECTED_LOCAL_UNIT_PRODUCTION_V1"

from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

FIRELINE_PARQUET = PROJECT_ROOT / "data" / "processed" / "fireline" / "fireline_qaqc_version_resolved.parquet"
OLD_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline_dynamic_250m"
OLD_SEGMENT_ROOT = OLD_ROOT / "segments"
OLD_ML_COHORT_PATH = OLD_ROOT / "ml_cohort" / "ml_cohort_primary_250m_50m_48h_v2.parquet"
NIROPS_SEARCH_ROOT = PROJECT_ROOT / "data" / "interim" / "joinability_audit" / "nirops_extracted"

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline_corrected_local_250m_10m"
CANONICAL_SHARD_ROOT = PROCESSED_ROOT / "canonical_parent_shards"
UNIT_ROOT = PROCESSED_ROOT / "units"
SAMPLE_ROOT = PROCESSED_ROOT / "sample_points"
TRAJECTORY_ROOT = PROCESSED_ROOT / "trajectories"
POINT_EVIDENCE_ROOT = PROCESSED_ROOT / "point_evidence"
UNIT_EVIDENCE_ROOT = PROCESSED_ROOT / "unit_evidence"
INCIDENT_META_ROOT = PROCESSED_ROOT / "incident_metadata"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "corrected_local_unit_production"

for p in [
    PROCESSED_ROOT, CANONICAL_SHARD_ROOT, UNIT_ROOT, SAMPLE_ROOT,
    TRAJECTORY_ROOT, POINT_EVIDENCE_ROOT, UNIT_EVIDENCE_ROOT,
    INCIDENT_META_ROOT, OUTPUT_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

assert FIRELINE_PARQUET.exists(), FIRELINE_PARQUET
assert OLD_SEGMENT_ROOT.exists(), OLD_SEGMENT_ROOT
assert NIROPS_SEARCH_ROOT.exists(), NIROPS_SEARCH_ROOT

SEGMENT_LENGTH_M = 250.0
FRAGMENT_GAP_M = 10.0
UNIT_SAMPLE_SPACING_M = 25.0

RASTER_RESOLUTION_M = 25.0
PRIMARY_TOLERANCE_M = 50.0
PRIMARY_MAX_GAP_HOURS = 48.0

MAX_DISTANCE_M = 250.0
FAR_DISTANCE_M = 251.0
GRID_PADDING_M = 300.0
MAX_SINGLE_GRID_CELLS = 20_000_000
TILE_CORE_SIZE_M = 50_000.0
TILE_HALO_M = 300.0

METRIC_CRS = "EPSG:5070"
ENGAGED_LABELS = ["Held", "Burned Over"]
MIN_NIROPS_OBSERVATIONS = 3

RESUME = True
MAX_INCIDENTS = None

UNIT_EVIDENCE_RULE = (
    "Held: any supported sample + zero conflict samples; "
    "Burned Over: any supported crossing sample"
)


In [ ]:

required = {
    "pandas": "pandas", "numpy": "numpy", "geopandas": "geopandas",
    "pyarrow": "pyarrow", "shapely": "shapely", "rasterio": "rasterio",
    "scipy": "scipy", "tqdm": "tqdm",
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import re, json, gc, time, math, hashlib
from collections import defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

from shapely.geometry import LineString, MultiLineString
from shapely.ops import substring
from scipy.spatial import cKDTree
from scipy.ndimage import distance_transform_edt
from rasterio.features import rasterize
from rasterio.transform import from_origin
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 220)

print("Python:", sys.executable)
print("GeoPandas:", gpd.__version__)
print("Shapely:", shapely.__version__)


## Configuration guard

In [ ]:

SCIENTIFIC_CONFIG = {
    "segment_length_m": SEGMENT_LENGTH_M,
    "fragment_gap_m": FRAGMENT_GAP_M,
    "unit_sample_spacing_m": UNIT_SAMPLE_SPACING_M,
    "raster_resolution_m": RASTER_RESOLUTION_M,
    "primary_tolerance_m": PRIMARY_TOLERANCE_M,
    "primary_max_gap_hours": PRIMARY_MAX_GAP_HOURS,
    "max_distance_m": MAX_DISTANCE_M,
    "grid_padding_m": GRID_PADDING_M,
    "max_single_grid_cells": MAX_SINGLE_GRID_CELLS,
    "tile_core_size_m": TILE_CORE_SIZE_M,
    "tile_halo_m": TILE_HALO_M,
    "metric_crs": METRIC_CRS,
    "engaged_labels": ENGAGED_LABELS,
    "minimum_nirops_observations": MIN_NIROPS_OBSERVATIONS,
    "unit_evidence_rule": UNIT_EVIDENCE_RULE,
}
PRODUCTION_CONFIG = {"notebook_build_id": NOTEBOOK_BUILD_ID, **SCIENTIFIC_CONFIG}
CONFIG_PATH = PROCESSED_ROOT / "PRODUCTION_CONFIG.json"

normalized = json.loads(json.dumps(SCIENTIFIC_CONFIG))
if CONFIG_PATH.exists():
    existing = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    existing_scientific = {k: v for k, v in existing.items() if k != "notebook_build_id"}
    if existing_scientific != normalized:
        raise RuntimeError("Existing 03L outputs use different scientific settings.")

CONFIG_PATH.write_text(json.dumps(PRODUCTION_CONFIG, indent=2), encoding="utf-8")
print(json.dumps(PRODUCTION_CONFIG, indent=2))


## Production incident inventory

In [ ]:

segment_files = sorted(OLD_SEGMENT_ROOT.glob("*.parquet"))
assert segment_files

inventory_rows = []
for path in tqdm(segment_files, desc="03G incident inventory"):
    df = pd.read_parquet(
        path,
        columns=["_irwin_norm", "IncidentName", "Year", "segment_uid", "_geom_hash"],
    )
    inventory_rows.append({
        "_irwin_norm": str(df["_irwin_norm"].iloc[0]),
        "IncidentName": df["IncidentName"].dropna().iloc[0] if df["IncidentName"].notna().any() else path.stem,
        "Year": df["Year"].dropna().iloc[0] if df["Year"].notna().any() else pd.NA,
        "old_segment_units": len(df),
        "parent_geometries": df["_geom_hash"].nunique(),
        "segment_path": str(path),
    })

incident_inventory = pd.DataFrame(inventory_rows).sort_values("old_segment_units").reset_index(drop=True)
if MAX_INCIDENTS is not None:
    incident_inventory = incident_inventory.head(MAX_INCIDENTS).copy()

PRODUCTION_IDS = incident_inventory["_irwin_norm"].astype(str).tolist()
print("Production incidents:", len(PRODUCTION_IDS))
print("Old 03G units:", f"{incident_inventory['old_segment_units'].sum():,}")
display(incident_inventory.head(20))



## One-time canonical parent sharding

The large canonical GeoParquet is touched only during this preparation stage. Exact `(_irwin_norm, _geom_hash)` keys from the 03G production shards define the required parent universe. Per-incident geometry shards are then reused by every restart.


In [ ]:

PARENT_KEY_CACHE = PROCESSED_ROOT / "production_parent_keys.parquet"
CANONICAL_COMPLETE_FLAG = PROCESSED_ROOT / "CANONICAL_PARENT_SHARDS_COMPLETE.json"

canonical_cache_ready = (
    RESUME
    and CANONICAL_COMPLETE_FLAG.exists()
    and all(
        (CANONICAL_SHARD_ROOT / f"{irwin}.parquet").exists()
        for irwin in PRODUCTION_IDS
    )
)

if canonical_cache_ready:
    print("Canonical parent shards already complete — reuse.")
else:
    parent_frames = []
    for path_text in tqdm(incident_inventory["segment_path"], desc="Collect parent keys"):
        d = pd.read_parquet(Path(path_text), columns=["_irwin_norm", "_geom_hash"])
        parent_frames.append(d.drop_duplicates())
    parent_keys = pd.concat(parent_frames, ignore_index=True).drop_duplicates().reset_index(drop=True)
    del parent_frames
    parent_keys["_irwin_norm"] = parent_keys["_irwin_norm"].astype(str)
    parent_keys.to_parquet(PARENT_KEY_CACHE, index=False)

    cols = [
        "_irwin_norm", "_geom_hash", "IncidentName", "Year",
        "FeatureCategory", "_engagement_norm", "geometry",
    ]
    t0 = time.perf_counter()
    try:
        canonical = gpd.read_parquet(
            FIRELINE_PARQUET,
            columns=cols,
            filters=[
                ("_irwin_norm", "in", PRODUCTION_IDS),
                ("_engagement_norm", "in", ENGAGED_LABELS),
            ],
        )
    except Exception as exc:
        print("Filtered canonical read unavailable; fallback:", type(exc).__name__, exc)
        canonical = gpd.read_parquet(FIRELINE_PARQUET, columns=cols)
        canonical = canonical[
            canonical["_irwin_norm"].astype(str).isin(PRODUCTION_IDS)
            & canonical["_engagement_norm"].isin(ENGAGED_LABELS)
        ].copy()

    crs = canonical.crs
    canonical["_irwin_norm"] = canonical["_irwin_norm"].astype(str)
    canonical = canonical.drop_duplicates(["_irwin_norm", "_geom_hash"])
    canonical = canonical.merge(
        parent_keys,
        on=["_irwin_norm", "_geom_hash"],
        how="inner",
        validate="one_to_one",
    )
    canonical = gpd.GeoDataFrame(canonical, geometry="geometry", crs=crs)
    canonical = canonical.to_crs(METRIC_CRS)

    if len(canonical) != len(parent_keys):
        raise RuntimeError(
            f"Canonical recovery incomplete: {len(canonical):,} rows vs {len(parent_keys):,} parent keys."
        )

    for irwin, group in tqdm(
        canonical.groupby("_irwin_norm", sort=False),
        total=canonical["_irwin_norm"].nunique(),
        desc="Write canonical shards",
    ):
        gpd.GeoDataFrame(group, geometry="geometry", crs=canonical.crs).to_parquet(
            CANONICAL_SHARD_ROOT / f"{irwin}.parquet",
            index=False,
        )

    summary = {
        "parent_keys": len(parent_keys),
        "canonical_rows": len(canonical),
        "incidents": canonical["_irwin_norm"].nunique(),
        "runtime_seconds": time.perf_counter() - t0,
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    CANONICAL_COMPLETE_FLAG.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(summary)
    del canonical, parent_keys
    gc.collect()


## Load NIROPS once

In [ ]:

def _norm_name(x):
    return re.sub(r"[^a-z0-9]", "", str(x).lower())

def find_column(columns, exact=(), contains=()):
    lookup = {_norm_name(c): c for c in columns}
    for x in exact:
        if _norm_name(x) in lookup:
            return lookup[_norm_name(x)]
    for token in contains:
        token = _norm_name(token)
        for key, original in lookup.items():
            if token in key:
                return original
    return None

def normalize_irwin(value):
    if pd.isna(value):
        return None
    value = re.sub(r"[^A-Z0-9]", "", str(value).strip().upper())
    return value or None

candidates = list(NIROPS_SEARCH_ROOT.rglob("NIROPS_2020_2024_R1_R6.shp"))
if not candidates:
    candidates = list(NIROPS_SEARCH_ROOT.rglob("*.shp"))
assert candidates
NIROPS_SHP = candidates[0]

nirops = gpd.read_file(NIROPS_SHP, engine="pyogrio")
NR_IRWIN = find_column(nirops.columns, exact=["IRWINID", "IRWIN_ID"], contains=["irwin"])
NR_TIME = find_column(nirops.columns, exact=["UTC", "DateUTC", "DateTimeUTC"], contains=["utc", "dateutc"])
assert NR_IRWIN is not None and NR_TIME is not None

nirops["_irwin_norm"] = nirops[NR_IRWIN].map(normalize_irwin)
nirops["_time"] = pd.to_datetime(nirops[NR_TIME], errors="coerce", utc=True)
nirops = nirops[
    nirops["_irwin_norm"].isin(PRODUCTION_IDS)
].dropna(subset=["_time"]).copy()
nirops_metric = nirops.to_crs(METRIC_CRS)

print("NIROPS rows:", f"{len(nirops_metric):,}")
print("NIROPS incidents:", nirops_metric["_irwin_norm"].nunique())


## Corrected-unit geometry helpers

In [ ]:

OLD_SEGMENT_COLUMNS = [
    "source_index", "_irwin_norm", "IncidentName", "Year", "FeatureCategory",
    "_engagement_norm", "_geom_hash", "_length_km_geom", "component_index",
    "segment_index", "segment_start_m", "segment_end_m",
    "actual_segment_length_m", "segment_uid", "segment_row_index",
]

def safe_substring(line, start_m, end_m):
    start_m = max(0.0, float(start_m))
    end_m = min(float(line.length), float(end_m))
    result = substring(line, start_m, end_m)
    if result.geom_type == "Point":
        c = result.coords[0]
        return LineString([c, c])
    return result

def aggregate_uid(irwin, geom_hash, component_indices):
    payload = (
        str(irwin) + "|" + str(geom_hash) + "|"
        + ",".join(str(int(x)) for x in component_indices)
    )
    digest = hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]
    return f"A::{irwin}::{geom_hash}::{digest}"

class UnionFind:
    def __init__(self, n):
        self.parent = np.arange(n, dtype=np.int32)
        self.rank = np.zeros(n, dtype=np.int8)

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = int(self.parent[x])
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

def endpoint_edges_10m(geometries):
    n = len(geometries)
    if n <= 1:
        return np.empty((0, 3), dtype=np.float64)

    endpoints = np.empty((2 * n, 2), dtype=np.float64)
    for i, geom in enumerate(geometries):
        c0 = geom.coords[0]
        c1 = geom.coords[-1]
        endpoints[2 * i] = c0[:2]
        endpoints[2 * i + 1] = c1[:2]

    endpoint_component = np.repeat(np.arange(n, dtype=np.int32), 2)
    pair_min = {}

    if n <= 4:
        for ea in range(2 * n):
            ca = int(endpoint_component[ea])
            for eb in range(ea + 1, 2 * n):
                cb = int(endpoint_component[eb])
                if ca == cb:
                    continue
                d = float(np.linalg.norm(endpoints[ea] - endpoints[eb]))
                if d > FRAGMENT_GAP_M:
                    continue
                a, b = sorted((ca, cb))
                key = (a, b)
                if key not in pair_min or d < pair_min[key]:
                    pair_min[key] = d
    else:
        tree = cKDTree(endpoints)
        for ea, eb in tree.query_pairs(r=FRAGMENT_GAP_M, output_type="ndarray"):
            ca = int(endpoint_component[ea])
            cb = int(endpoint_component[eb])
            if ca == cb:
                continue
            if ca > cb:
                ca, cb = cb, ca
            d = float(np.linalg.norm(endpoints[ea] - endpoints[eb]))
            key = (ca, cb)
            if key not in pair_min or d < pair_min[key]:
                pair_min[key] = d

    if not pair_min:
        return np.empty((0, 3), dtype=np.float64)

    return np.asarray(
        [(a, b, d) for (a, b), d in pair_min.items()],
        dtype=np.float64,
    )

def graph_adjacency(n, edges):
    adjacency = [[] for _ in range(n)]
    uf = UnionFind(n)
    for a_raw, b_raw, d_raw in edges:
        a, b, d = int(a_raw), int(b_raw), float(d_raw)
        adjacency[a].append((b, d))
        adjacency[b].append((a, d))
        uf.union(a, b)

    groups = defaultdict(list)
    for i in range(n):
        groups[uf.find(i)].append(i)
    return adjacency, list(groups.values())

def pack_connected(component_ids, lengths, adjacency, connected_indices):
    remaining = set(connected_indices)
    packed = []

    while remaining:
        seed = min(remaining, key=lambda i: int(component_ids[i]))
        selected = [seed]
        selected_set = {seed}
        remaining.remove(seed)
        current_length = float(lengths[seed])
        used_gaps = []

        while True:
            best = None
            for current in selected_set:
                for neighbor, gap in adjacency[current]:
                    if neighbor not in remaining:
                        continue
                    if current_length + float(lengths[neighbor]) > SEGMENT_LENGTH_M + 1e-9:
                        continue
                    candidate = (float(gap), int(component_ids[neighbor]), neighbor)
                    if best is None or candidate < best:
                        best = candidate

            if best is None:
                break

            gap, _, chosen = best
            selected.append(chosen)
            selected_set.add(chosen)
            remaining.remove(chosen)
            current_length += float(lengths[chosen])
            used_gaps.append(gap)

        packed.append((selected, used_gaps))

    return packed



## Fast incident unit construction

The canonical parents are exploded into primitive line components with GeoPandas in one vectorized operation. Only parent geometries containing **two or more short components** enter Python graph loops.

This avoids the tens-of-thousands-of-parent graph loop that made the exploratory 03K audit slow.


In [ ]:

def build_incident_units(old_segments, canonical_parents):
    old_segments = old_segments.copy()
    old_segments["_geom_hash"] = old_segments["_geom_hash"].astype(str)

    canonical = canonical_parents.to_crs(METRIC_CRS).copy()
    canonical["_geom_hash"] = canonical["_geom_hash"].astype(str)
    canonical["_parent_row"] = np.arange(len(canonical), dtype=np.int32)

    # Vectorized component explosion.
    exploded = (
        canonical
        .set_index("_parent_row")
        .explode(index_parts=True)
        .reset_index()
        .rename(columns={"level_1": "component_index"})
    )

    if "component_index" not in exploded.columns:
        # GeoPandas may name the second reset index column differently.
        possible = [
            c for c in exploded.columns
            if c not in canonical.columns and c != "_parent_row"
        ]
        if possible:
            exploded = exploded.rename(columns={possible[0]: "component_index"})

    exploded["component_index"] = exploded["component_index"].astype(np.int32)

    component_summary = (
        old_segments.groupby(
            ["_geom_hash", "component_index"],
            as_index=False,
        )
        .agg(
            component_length_m=("actual_segment_length_m", "sum"),
            old_component_units=("segment_uid", "size"),
            source_index=("source_index", "first"),
            old_component_segment_uid=("segment_uid", "first"),
        )
    )

    components = exploded.merge(
        component_summary,
        on=["_geom_hash", "component_index"],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    if not (components["_merge"] == "both").all():
        counts = components["_merge"].value_counts().to_dict()
        raise RuntimeError(f"Canonical/component mismatch: {counts}")

    components = gpd.GeoDataFrame(
        components.drop(columns="_merge"),
        geometry="geometry",
        crs=canonical.crs,
    )

    components["canonical_length_m"] = components.geometry.length
    abs_diff = (
        components["canonical_length_m"]
        - components["component_length_m"]
    ).abs()

    rel_diff = abs_diff / components["canonical_length_m"].clip(lower=1e-9)
    inconsistent = (abs_diff > 1.0) & (rel_diff > 0.01)

    if inconsistent.any():
        bad = components.loc[
            inconsistent,
            ["_geom_hash", "component_index", "canonical_length_m", "component_length_m"],
        ].head(20)
        raise RuntimeError(
            "Canonical component length mismatch.\n" + bad.to_string(index=False)
        )

    components["is_short_component"] = (
        components["component_length_m"] < SEGMENT_LENGTH_M
    )

    # ------------------------------------------------------------
    # A. Long components: keep exact 03G segment boundaries.
    # ------------------------------------------------------------
    long_component_keys = components.loc[
        ~components["is_short_component"],
        ["_geom_hash", "component_index", "geometry"],
    ].rename(columns={"geometry": "component_geometry"})

    long_rows = old_segments.merge(
        long_component_keys,
        on=["_geom_hash", "component_index"],
        how="inner",
        validate="many_to_one",
    )

    long_geometries = [
        safe_substring(g, s, e)
        for g, s, e in zip(
            long_rows["component_geometry"],
            long_rows["segment_start_m"],
            long_rows["segment_end_m"],
        )
    ]

    long_units = gpd.GeoDataFrame(
        {
            "unit_uid": "L::" + long_rows["segment_uid"].astype(str),
            "unit_origin": "unchanged_long_component_segment",
            "_irwin_norm": long_rows["_irwin_norm"].astype(str).to_numpy(),
            "IncidentName": long_rows["IncidentName"].to_numpy(),
            "Year": long_rows["Year"].to_numpy(),
            "FeatureCategory": long_rows["FeatureCategory"].to_numpy(),
            "_engagement_norm": long_rows["_engagement_norm"].to_numpy(),
            "_geom_hash": long_rows["_geom_hash"].astype(str).to_numpy(),
            "source_index": long_rows["source_index"].to_numpy(),
            "primary_component_index": long_rows["component_index"].astype(np.int32).to_numpy(),
            "constituent_components": 1,
            "constituent_component_indices_json": long_rows["component_index"].astype(int).map(
                lambda x: json.dumps([x])
            ).to_numpy(),
            "constituent_old_segment_uids_json": long_rows["segment_uid"].astype(str).map(
                lambda x: json.dumps([x])
            ).to_numpy(),
            "actual_line_length_m": np.asarray([g.length for g in long_geometries], dtype=float),
            "max_gap_used_m": 0.0,
            "mean_gap_used_m": 0.0,
        },
        geometry=long_geometries,
        crs=METRIC_CRS,
    )

    # ------------------------------------------------------------
    # B. Short components.
    # ------------------------------------------------------------
    short = components[components["is_short_component"]].copy()

    if len(short):
        short_parent_count = (
            short.groupby("_geom_hash")["component_index"]
            .transform("size")
            .astype(np.int32)
        )
        short["short_components_in_parent"] = short_parent_count
    else:
        short["short_components_in_parent"] = pd.Series(dtype=np.int32)

    # Singleton short-component parents: no KD-tree needed.
    singleton = short[short["short_components_in_parent"] == 1].copy()

    singleton_units = gpd.GeoDataFrame(
        {
            "unit_uid": [
                aggregate_uid(irwin, geom_hash, [component_index])
                for irwin, geom_hash, component_index in zip(
                    singleton["_irwin_norm"],
                    singleton["_geom_hash"],
                    singleton["component_index"],
                )
            ],
            "unit_origin": "locally_aggregated_short_components",
            "_irwin_norm": singleton["_irwin_norm"].astype(str).to_numpy(),
            "IncidentName": singleton["IncidentName"].to_numpy(),
            "Year": singleton["Year"].to_numpy(),
            "FeatureCategory": singleton["FeatureCategory"].to_numpy(),
            "_engagement_norm": singleton["_engagement_norm"].to_numpy(),
            "_geom_hash": singleton["_geom_hash"].astype(str).to_numpy(),
            "source_index": singleton["source_index"].to_numpy(),
            "primary_component_index": singleton["component_index"].astype(np.int32).to_numpy(),
            "constituent_components": 1,
            "constituent_component_indices_json": singleton["component_index"].astype(int).map(
                lambda x: json.dumps([x])
            ).to_numpy(),
            "constituent_old_segment_uids_json": singleton["old_component_segment_uid"].astype(str).map(
                lambda x: json.dumps([x])
            ).to_numpy(),
            "actual_line_length_m": singleton["component_length_m"].to_numpy(dtype=float),
            "max_gap_used_m": 0.0,
            "mean_gap_used_m": 0.0,
        },
        geometry=singleton.geometry.to_numpy(),
        crs=METRIC_CRS,
    )

    # Multi-short parents: only these enter graph loops.
    multi = short[short["short_components_in_parent"] >= 2].copy()
    multi_rows = []
    graph_parent_count = 0
    no_edge_parent_count = 0

    for geom_hash, group in multi.groupby("_geom_hash", sort=False):
        group = group.sort_values("component_index").reset_index(drop=True)
        geoms = group.geometry.tolist()
        component_ids = group["component_index"].to_numpy(dtype=np.int32)
        lengths = group["component_length_m"].to_numpy(dtype=np.float64)

        edges = endpoint_edges_10m(geoms)

        if len(edges) == 0:
            no_edge_parent_count += 1
            packed_groups = [([i], []) for i in range(len(group))]
        else:
            graph_parent_count += 1
            adjacency, connected_groups = graph_adjacency(len(group), edges)
            packed_groups = []
            for connected_indices in connected_groups:
                packed_groups.extend(
                    pack_connected(
                        component_ids=component_ids,
                        lengths=lengths,
                        adjacency=adjacency,
                        connected_indices=connected_indices,
                    )
                )

        for selected_local, used_gaps in packed_groups:
            selected = group.iloc[selected_local]
            selected_geoms = selected.geometry.tolist()
            component_indices = selected["component_index"].astype(int).tolist()
            old_uids = selected["old_component_segment_uid"].astype(str).tolist()

            geometry = (
                selected_geoms[0]
                if len(selected_geoms) == 1
                else MultiLineString(selected_geoms)
            )

            first = selected.iloc[0]
            multi_rows.append({
                "unit_uid": aggregate_uid(
                    first["_irwin_norm"],
                    first["_geom_hash"],
                    component_indices,
                ),
                "unit_origin": "locally_aggregated_short_components",
                "_irwin_norm": str(first["_irwin_norm"]),
                "IncidentName": first["IncidentName"],
                "Year": first["Year"],
                "FeatureCategory": first["FeatureCategory"],
                "_engagement_norm": first["_engagement_norm"],
                "_geom_hash": str(first["_geom_hash"]),
                "source_index": first["source_index"],
                "primary_component_index": int(component_indices[0]),
                "constituent_components": len(component_indices),
                "constituent_component_indices_json": json.dumps(component_indices),
                "constituent_old_segment_uids_json": json.dumps(old_uids),
                "actual_line_length_m": float(selected["component_length_m"].sum()),
                "max_gap_used_m": float(max(used_gaps)) if used_gaps else 0.0,
                "mean_gap_used_m": float(np.mean(used_gaps)) if used_gaps else 0.0,
                "geometry": geometry,
            })

    multi_units = (
        gpd.GeoDataFrame(multi_rows, geometry="geometry", crs=METRIC_CRS)
        if multi_rows
        else singleton_units.iloc[0:0].copy()
    )

    units = pd.concat(
        [long_units, singleton_units, multi_units],
        ignore_index=True,
        sort=False,
    )
    units = gpd.GeoDataFrame(units, geometry="geometry", crs=METRIC_CRS)

    units = units.sort_values(
        ["_geom_hash", "primary_component_index", "unit_uid"]
    ).reset_index(drop=True)
    units["unit_row_index"] = np.arange(len(units), dtype=np.int32)

    if units["unit_uid"].duplicated().any():
        raise RuntimeError("Duplicate corrected unit_uid.")
    if units.geometry.is_empty.any() or units.geometry.isna().any():
        raise RuntimeError("Empty/missing corrected unit geometry.")
    if (units["actual_line_length_m"] <= 0).any():
        raise RuntimeError("Non-positive corrected unit length.")
    if (units["actual_line_length_m"] > SEGMENT_LENGTH_M + 1.0).any():
        raise RuntimeError("Corrected unit exceeds 250 m actual-line-length limit.")

    metadata = {
        "canonical_components": int(len(components)),
        "short_components": int(len(short)),
        "singleton_short_parent_components": int(len(singleton)),
        "multi_short_parent_components": int(len(multi)),
        "multi_short_parents": int(multi["_geom_hash"].nunique()),
        "graph_short_parents": int(graph_parent_count),
        "multi_short_parents_no_10m_edges": int(no_edge_parent_count),
        "max_component_length_difference_m": float(abs_diff.max()) if len(abs_diff) else 0.0,
    }

    return units, metadata



## Fast within-unit sampling

Most corrected units are single `LineString`s. Those units are sampled vectorially with Shapely. Only aggregated `MultiLineString` units use a small Python loop over constituent parts.

For each geometry part of length \(L\):

\[
N=\max(1,\lceil L/25\rceil)
\]

with sample points at equal-length bin centers.


In [ ]:

def build_sample_points(units):
    geometry_types = units.geometry.geom_type.to_numpy()
    line_mask = geometry_types == "LineString"
    line_indices = np.flatnonzero(line_mask)
    multi_indices = np.flatnonzero(~line_mask)

    frames = []

    # ------------------------------------------------------------
    # Vectorized LineString sampling. n is at most ~10 for a
    # <=250 m unit, so group by n and interpolate in bulk.
    # ------------------------------------------------------------
    if len(line_indices):
        line_geoms = units.geometry.iloc[line_indices].to_numpy()
        line_lengths = units["actual_line_length_m"].iloc[line_indices].to_numpy(dtype=float)
        n_points = np.maximum(
            1,
            np.ceil(line_lengths / UNIT_SAMPLE_SPACING_M).astype(np.int32),
        )

        for n in np.unique(n_points):
            local = np.flatnonzero(n_points == n)
            unit_idx = line_indices[local]
            geoms = line_geoms[local]
            lengths = line_lengths[local]

            repeated_geoms = np.repeat(geoms, n)
            repeated_units = np.repeat(unit_idx, n)
            point_order = np.tile(np.arange(n, dtype=np.int16), len(local))
            fractions = (point_order.astype(float) + 0.5) / float(n)
            distances = np.repeat(lengths, n) * fractions

            try:
                pts = shapely.line_interpolate_point(repeated_geoms, distances)
                xs = shapely.get_x(pts)
                ys = shapely.get_y(pts)
            except Exception:
                pts = [
                    g.interpolate(float(d))
                    for g, d in zip(repeated_geoms, distances)
                ]
                xs = np.asarray([p.x for p in pts], dtype=float)
                ys = np.asarray([p.y for p in pts], dtype=float)

            frames.append(pd.DataFrame({
                "unit_row_index": repeated_units.astype(np.int32),
                "part_index": np.zeros(len(repeated_units), dtype=np.int16),
                "point_index_within_part": point_order,
                "x_m": xs,
                "y_m": ys,
            }))

    # ------------------------------------------------------------
    # MultiLineString sampling. Every constituent part receives
    # at least one sample point.
    # ------------------------------------------------------------
    multi_rows = []
    for unit_idx in multi_indices:
        geom = units.geometry.iloc[unit_idx]
        parts = list(geom.geoms) if geom.geom_type == "MultiLineString" else [geom]
        for part_index, part in enumerate(parts):
            length = float(part.length)
            if length <= 0:
                continue
            n = max(1, int(math.ceil(length / UNIT_SAMPLE_SPACING_M)))
            distances = (np.arange(n, dtype=float) + 0.5) * length / n
            for point_index, distance in enumerate(distances):
                p = part.interpolate(float(distance))
                multi_rows.append({
                    "unit_row_index": int(unit_idx),
                    "part_index": int(part_index),
                    "point_index_within_part": int(point_index),
                    "x_m": float(p.x),
                    "y_m": float(p.y),
                })

    if multi_rows:
        frames.append(pd.DataFrame(multi_rows))

    samples = pd.concat(frames, ignore_index=True)
    samples = samples.merge(
        units[
            ["unit_row_index", "unit_uid", "_irwin_norm", "_engagement_norm"]
        ],
        on="unit_row_index",
        how="left",
        validate="many_to_one",
    )
    samples["sample_row_index"] = np.arange(len(samples), dtype=np.int32)

    return samples[
        [
            "sample_row_index", "unit_row_index", "unit_uid",
            "_irwin_norm", "_engagement_norm", "part_index",
            "point_index_within_part", "x_m", "y_m",
        ]
    ]


## NIROPS progression and adaptive raster helpers

In [ ]:

def prepare_progression(incident_nirops):
    progression = (
        incident_nirops[["_time", "geometry"]]
        .dropna(subset=["_time", "geometry"])
        .sort_values("_time")
        .copy()
    )

    if progression.empty:
        return progression

    if not progression["_time"].is_unique:
        rows = []
        for timestamp, group in progression.groupby("_time"):
            geometry = (
                group.geometry.iloc[0]
                if len(group) == 1
                else group.geometry.union_all()
            )
            rows.append({"_time": timestamp, "geometry": geometry})
        progression = gpd.GeoDataFrame(
            rows, geometry="geometry", crs=incident_nirops.crs
        )

    return progression.sort_values("_time").reset_index(drop=True)

def bounds_intersect(a, b):
    return not (
        a[2] < b[0] or a[0] > b[2]
        or a[3] < b[1] or a[1] > b[3]
    )

def make_region(point_indices, x, y, minx, miny, maxx, maxy, mode, region_id):
    resolution = RASTER_RESOLUTION_M
    width = int(np.ceil((maxx - minx) / resolution))
    height = int(np.ceil((maxy - miny) / resolution))
    transform = from_origin(minx, maxy, resolution, resolution)

    lx = x[point_indices]
    ly = y[point_indices]
    cols = np.floor((lx - minx) / resolution).astype(np.int64)
    rows = np.floor((maxy - ly) / resolution).astype(np.int64)
    cols = np.clip(cols, 0, width - 1)
    rows = np.clip(rows, 0, height - 1)

    return {
        "mode": mode,
        "region_id": region_id,
        "point_indices": np.asarray(point_indices, dtype=np.int64),
        "point_rows": rows,
        "point_cols": cols,
        "width": width,
        "height": height,
        "cells": width * height,
        "transform": transform,
        "bounds": (float(minx), float(miny), float(maxx), float(maxy)),
    }

def build_processing_regions(sample_points):
    x = sample_points["x_m"].to_numpy(dtype=np.float64)
    y = sample_points["y_m"].to_numpy(dtype=np.float64)
    r = RASTER_RESOLUTION_M

    minx = np.floor((x.min() - GRID_PADDING_M) / r) * r
    maxx = np.ceil((x.max() + GRID_PADDING_M) / r) * r
    miny = np.floor((y.min() - GRID_PADDING_M) / r) * r
    maxy = np.ceil((y.max() + GRID_PADDING_M) / r) * r

    width = int(np.ceil((maxx - minx) / r))
    height = int(np.ceil((maxy - miny) / r))
    cells = width * height

    if cells <= MAX_SINGLE_GRID_CELLS:
        return {
            "mode": "single",
            "single_grid_cells": cells,
            "regions": [
                make_region(
                    np.arange(len(sample_points)),
                    x, y, minx, miny, maxx, maxy,
                    "single", "single",
                )
            ],
        }

    tile_x = np.floor(x / TILE_CORE_SIZE_M).astype(np.int64)
    tile_y = np.floor(y / TILE_CORE_SIZE_M).astype(np.int64)
    unique_tiles = np.unique(np.column_stack([tile_x, tile_y]), axis=0)

    regions = []
    for tx, ty in unique_tiles:
        tx, ty = int(tx), int(ty)
        idx = np.flatnonzero((tile_x == tx) & (tile_y == ty))
        core_minx = tx * TILE_CORE_SIZE_M
        core_maxx = core_minx + TILE_CORE_SIZE_M
        core_miny = ty * TILE_CORE_SIZE_M
        core_maxy = core_miny + TILE_CORE_SIZE_M

        regions.append(
            make_region(
                idx, x, y,
                core_minx - TILE_HALO_M,
                core_miny - TILE_HALO_M,
                core_maxx + TILE_HALO_M,
                core_maxy + TILE_HALO_M,
                "tile", f"tile_{tx}_{ty}",
            )
        )

    return {
        "mode": "tiled",
        "single_grid_cells": cells,
        "regions": regions,
    }

def sample_region_signed_distance(polygon, polygon_bounds, region):
    n = len(region["point_indices"])

    if not bounds_intersect(polygon_bounds, region["bounds"]):
        return np.full(n, FAR_DISTANCE_M, dtype=np.float32)

    mask = rasterize(
        [(polygon, 1)],
        out_shape=(region["height"], region["width"]),
        transform=region["transform"],
        fill=0,
        default_value=1,
        dtype="uint8",
        all_touched=False,
    )

    rows = region["point_rows"]
    cols = region["point_cols"]

    if not mask.any():
        return np.full(n, FAR_DISTANCE_M, dtype=np.float32)
    if mask.all():
        return np.full(n, -FAR_DISTANCE_M, dtype=np.float32)

    point_inside = mask[rows, cols] == 1
    signed = np.full(n, FAR_DISTANCE_M, dtype=np.float32)

    if point_inside.any():
        inside_distance = distance_transform_edt(
            mask,
            sampling=RASTER_RESOLUTION_M,
        )
        signed[point_inside] = -np.minimum(
            inside_distance[rows[point_inside], cols[point_inside]],
            FAR_DISTANCE_M,
        ).astype(np.float32)
        del inside_distance

    point_outside = ~point_inside
    if point_outside.any():
        outside_distance = distance_transform_edt(
            mask == 0,
            sampling=RASTER_RESOLUTION_M,
        )
        signed[point_outside] = np.minimum(
            outside_distance[rows[point_outside], cols[point_outside]],
            FAR_DISTANCE_M,
        ).astype(np.float32)
        del outside_distance

    del mask
    return signed

def build_incident_trajectory(sample_points, progression, incident_name):
    region_info = build_processing_regions(sample_points)

    signed = np.full(
        (len(sample_points), len(progression)),
        FAR_DISTANCE_M,
        dtype=np.float32,
    )

    inner = tqdm(
        progression[["_time", "geometry"]].itertuples(index=False, name=None),
        total=len(progression),
        desc=f"{str(incident_name)[:24]} NIROPS",
        leave=False,
    )

    for time_index, (timestamp, polygon) in enumerate(inner):
        polygon_bounds = polygon.bounds
        for region in region_info["regions"]:
            signed[region["point_indices"], time_index] = (
                sample_region_signed_distance(
                    polygon=polygon,
                    polygon_bounds=polygon_bounds,
                    region=region,
                )
            )

    return signed, region_info


## Vectorized sample-point temporal classifier

In [ ]:

def first_true_index(mask):
    result = np.full(mask.shape[0], -1, dtype=np.int32)
    if mask.ndim != 2 or mask.shape[1] == 0:
        return result
    has_true = mask.any(axis=1)
    result[has_true] = mask[has_true].argmax(axis=1)
    return result

def classify_sample_points(labels, signed, times):
    labels = np.asarray(labels, dtype=object)
    n_points, n_times = signed.shape

    near = np.abs(signed) <= PRIMARY_TOLERANCE_M
    outside = signed > PRIMARY_TOLERANCE_M
    inside = signed < -PRIMARY_TOLERANCE_M

    gaps_hours = np.asarray(
        [
            (times[j + 1] - times[j]).total_seconds() / 3600.0
            for j in range(n_times - 1)
        ],
        dtype=np.float32,
    )
    short_gap = gaps_hours <= PRIMARY_MAX_GAP_HOURS
    long_gap = ~short_gap

    cross_any = outside[:, :-1] & inside[:, 1:]
    cross_short = cross_any & short_gap[np.newaxis, :]
    cross_long = cross_any & long_gap[np.newaxis, :]

    outside_to_near = outside[:, :-1] & near[:, 1:]
    near_short = outside_to_near & short_gap[np.newaxis, :]
    near_long = outside_to_near & long_gap[np.newaxis, :]

    first_cross_short = first_true_index(cross_short)
    first_cross_long = first_true_index(cross_long)
    first_near_long = first_true_index(near_long)

    first_inside_observation = inside[:, 0]
    ever_inside = inside.any(axis=1)
    ever_near = near.any(axis=1)

    last_near = np.full(n_points, -1, dtype=np.int32)
    has_near = near.any(axis=1)
    last_near[has_near] = (
        n_times - 1 - near[has_near, ::-1].argmax(axis=1)
    )

    suffix_inside = np.maximum.accumulate(
        inside[:, ::-1],
        axis=1,
    )[:, ::-1].astype(bool)

    valid_hold = np.zeros_like(near_short, dtype=bool)
    if n_times >= 3:
        transition_j = np.arange(n_times - 1)
        has_followup = transition_j <= n_times - 3
        no_inside_from_challenge = ~suffix_inside[:, 1:]
        valid_hold = (
            near_short
            & has_followup[np.newaxis, :]
            & no_inside_from_challenge
        )

    first_valid_hold = first_true_index(valid_hold)

    evidence = np.full(
        n_points,
        "insufficient_temporal_evidence",
        dtype=object,
    )
    reason = np.full(
        n_points,
        "no_observed_challenge",
        dtype=object,
    )
    event_start_idx = np.full(n_points, -1, dtype=np.int32)
    event_end_idx = np.full(n_points, -1, dtype=np.int32)

    # Burned Over
    burned = labels == "Burned Over"
    supported = burned & (first_cross_short >= 0)
    evidence[supported] = "temporally_supported"
    reason[supported] = "outside_to_inside_transition"
    event_start_idx[supported] = first_cross_short[supported]
    event_end_idx[supported] = first_cross_short[supported] + 1

    first_inside = burned & ~supported & first_inside_observation
    reason[first_inside] = "first_observation_already_inside"

    long_cross = burned & ~supported & ~first_inside & (first_cross_long >= 0)
    reason[long_cross] = "outside_to_inside_gap_too_large"

    inside_other = (
        burned & ~supported & ~first_inside & ~long_cross & ever_inside
    )
    reason[inside_other] = "inside_without_observed_transition"

    last_challenge = burned & ~ever_inside & (last_near == n_times - 1)
    reason[last_challenge] = "challenge_at_last_observation"

    conflict = burned & ~ever_inside & ever_near & ~last_challenge
    evidence[conflict] = "trajectory_conflict"
    reason[conflict] = "approached_but_never_inside"

    # Held
    held = labels == "Held"

    held_cross = held & (first_cross_short >= 0)
    evidence[held_cross] = "trajectory_conflict"
    reason[held_cross] = "held_but_crossed_inside"
    event_start_idx[held_cross] = first_cross_short[held_cross]
    event_end_idx[held_cross] = first_cross_short[held_cross] + 1

    held_first_inside = held & ~held_cross & first_inside_observation
    evidence[held_first_inside] = "trajectory_conflict"
    reason[held_first_inside] = "first_observation_already_inside"

    held_supported = (
        held
        & ~held_cross
        & ~held_first_inside
        & (first_valid_hold >= 0)
    )
    evidence[held_supported] = "temporally_supported"
    reason[held_supported] = "outside_to_near_without_later_engulfment"
    event_start_idx[held_supported] = first_valid_hold[held_supported]
    event_end_idx[held_supported] = first_valid_hold[held_supported] + 1

    held_inside_other = (
        held
        & ~held_cross
        & ~held_first_inside
        & ~held_supported
        & ever_inside
    )
    evidence[held_inside_other] = "trajectory_conflict"
    reason[held_inside_other] = "sample_point_observed_inside"

    held_last = (
        held
        & ~held_cross
        & ~held_first_inside
        & ~held_supported
        & ~ever_inside
        & (last_near == n_times - 1)
    )
    reason[held_last] = "challenge_at_last_observation"

    held_long = (
        held
        & ~held_cross
        & ~held_first_inside
        & ~held_supported
        & ~ever_inside
        & ~held_last
        & (first_near_long >= 0)
    )
    reason[held_long] = "approach_gap_too_large"

    held_near_other = (
        held
        & ~held_cross
        & ~held_first_inside
        & ~held_supported
        & ~ever_inside
        & ~held_last
        & ~held_long
        & ever_near
    )
    reason[held_near_other] = "near_without_resolved_approach"

    return pd.DataFrame({
        "point_evidence": evidence,
        "point_diagnostic_reason": reason,
        "event_start_idx": event_start_idx,
        "event_end_idx": event_end_idx,
        "ever_inside": ever_inside,
        "ever_near": ever_near,
        "min_signed_distance_m": np.min(signed, axis=1),
        "min_abs_distance_m": np.min(np.abs(signed), axis=1),
    })



## Vectorized unit-level evidence aggregation

This stage uses Pandas groupby operations rather than looping over every corrected unit.


In [ ]:

def aggregate_unit_evidence(units, sample_points, point_evidence, times):
    point_table = pd.concat(
        [
            sample_points.reset_index(drop=True),
            point_evidence.reset_index(drop=True),
        ],
        axis=1,
    )

    point_table["_is_supported"] = (
        point_table["point_evidence"] == "temporally_supported"
    ).astype(np.int16)
    point_table["_is_conflict"] = (
        point_table["point_evidence"] == "trajectory_conflict"
    ).astype(np.int16)
    point_table["_is_insufficient"] = (
        point_table["point_evidence"] == "insufficient_temporal_evidence"
    ).astype(np.int16)

    summary = (
        point_table.groupby("unit_row_index", as_index=False)
        .agg(
            sample_points=("sample_row_index", "size"),
            supported_points=("_is_supported", "sum"),
            conflict_points=("_is_conflict", "sum"),
            insufficient_points=("_is_insufficient", "sum"),
            point_evidence_states=("point_evidence", "nunique"),
            min_signed_distance_m=("min_signed_distance_m", "min"),
            min_abs_distance_m=("min_abs_distance_m", "min"),
        )
    )

    summary["supported_fraction"] = (
        summary["supported_points"] / summary["sample_points"]
    )
    summary["conflict_fraction"] = (
        summary["conflict_points"] / summary["sample_points"]
    )
    summary["insufficient_fraction"] = (
        summary["insufficient_points"] / summary["sample_points"]
    )
    summary["all_same_point_evidence"] = (
        summary["point_evidence_states"] == 1
    )
    summary["any_supported"] = summary["supported_points"] > 0
    summary["majority_supported"] = (
        summary["supported_points"] > summary["sample_points"] / 2.0
    )
    summary["all_supported"] = (
        summary["supported_points"] == summary["sample_points"]
    )
    summary["any_conflict"] = summary["conflict_points"] > 0

    # Earliest supported challenge interval for each unit.
    events = point_table[
        (point_table["_is_supported"] == 1)
        & (point_table["event_end_idx"] >= 0)
    ][
        ["unit_row_index", "event_start_idx", "event_end_idx"]
    ].copy()

    if len(events):
        events = (
            events.sort_values(
                ["unit_row_index", "event_end_idx", "event_start_idx"]
            )
            .drop_duplicates("unit_row_index", keep="first")
        )
        events["challenge_interval_start_utc"] = [
            times[int(i)] for i in events["event_start_idx"]
        ]
        events["challenge_interval_end_utc"] = [
            times[int(i)] for i in events["event_end_idx"]
        ]
        events["challenge_interval_hours"] = (
            (
                pd.to_datetime(events["challenge_interval_end_utc"], utc=True)
                - pd.to_datetime(events["challenge_interval_start_utc"], utc=True)
            ).dt.total_seconds()
            / 3600.0
        )
    else:
        events = pd.DataFrame(
            columns=[
                "unit_row_index",
                "challenge_interval_start_utc",
                "challenge_interval_end_utc",
                "challenge_interval_hours",
            ]
        )

    unit_columns = [
        "unit_row_index", "unit_uid", "_irwin_norm", "IncidentName", "Year",
        "FeatureCategory", "_engagement_norm", "_geom_hash", "source_index",
        "unit_origin", "actual_line_length_m", "constituent_components",
        "max_gap_used_m", "mean_gap_used_m",
    ]

    result = (
        pd.DataFrame(units.drop(columns="geometry"))[unit_columns]
        .merge(summary, on="unit_row_index", how="left", validate="one_to_one")
        .merge(
            events[
                [
                    "unit_row_index",
                    "challenge_interval_start_utc",
                    "challenge_interval_end_utc",
                    "challenge_interval_hours",
                ]
            ],
            on="unit_row_index",
            how="left",
            validate="one_to_one",
        )
    )

    held = result["_engagement_norm"] == "Held"
    burned = result["_engagement_norm"] == "Burned Over"

    result["unit_evidence"] = "insufficient_temporal_evidence"
    result["unit_diagnostic_reason"] = "no_supported_sample_point"

    held_conflict = held & result["any_conflict"]
    result.loc[held_conflict, "unit_evidence"] = "trajectory_conflict"
    result.loc[held_conflict, "unit_diagnostic_reason"] = (
        "held_any_sample_point_conflict"
    )

    held_supported = held & ~result["any_conflict"] & result["any_supported"]
    result.loc[held_supported, "unit_evidence"] = "temporally_supported"
    result.loc[held_supported, "unit_diagnostic_reason"] = (
        "held_supported_approach_no_sample_point_conflict"
    )

    burned_supported = burned & result["any_supported"]
    result.loc[burned_supported, "unit_evidence"] = "temporally_supported"
    result.loc[burned_supported, "unit_diagnostic_reason"] = (
        "burned_over_any_supported_crossing_point"
    )

    burned_conflict = burned & ~result["any_supported"] & result["any_conflict"]
    result.loc[burned_conflict, "unit_evidence"] = "trajectory_conflict"
    result.loc[burned_conflict, "unit_diagnostic_reason"] = (
        "burned_over_no_crossing_but_point_conflict"
    )

    # Challenge interval is meaningful only for supported units.
    not_supported = result["unit_evidence"] != "temporally_supported"
    result.loc[
        not_supported,
        [
            "challenge_interval_start_utc",
            "challenge_interval_end_utc",
            "challenge_interval_hours",
        ],
    ] = [pd.NaT, pd.NaT, np.nan]

    assert result["unit_uid"].is_unique

    # Drop temporary count flags before saving point evidence.
    point_table = point_table.drop(
        columns=["_is_supported", "_is_conflict", "_is_insufficient"]
    )

    return result, point_table


## Resume paths

In [ ]:

def incident_paths(irwin):
    return {
        "canonical": CANONICAL_SHARD_ROOT / f"{irwin}.parquet",
        "units": UNIT_ROOT / f"{irwin}.parquet",
        "samples": SAMPLE_ROOT / f"{irwin}.parquet",
        "trajectory": TRAJECTORY_ROOT / f"{irwin}.npz",
        "point_evidence": POINT_EVIDENCE_ROOT / f"{irwin}.parquet",
        "unit_evidence": UNIT_EVIDENCE_ROOT / f"{irwin}.parquet",
        "metadata": INCIDENT_META_ROOT / f"{irwin}.json",
        "error": INCIDENT_META_ROOT / f"{irwin}_ERROR.txt",
    }

def incident_complete(irwin):
    p = incident_paths(irwin)
    return all(
        p[key].exists()
        for key in [
            "units", "samples", "trajectory",
            "point_evidence", "unit_evidence", "metadata",
        ]
    )



## Full production loop

The loop is sequential by incident because raster distance transforms are memory-intensive. Parallel incidents usually increase memory pressure more than they improve throughput.

Every expensive stage is cached. If the notebook is interrupted, rerun from the beginning; completed incident stages are reused.


In [ ]:

import traceback

RUN_LOG_PATH = OUTPUT_ROOT / "PRODUCTION_RUN_LOG.csv"
run_log_rows = []

outer = tqdm(
    incident_inventory.itertuples(index=False, name=None),
    total=len(incident_inventory),
    desc="03L production incidents",
)
inventory_columns = list(incident_inventory.columns)

for values in outer:
    inventory_row = dict(zip(inventory_columns, values))
    irwin = str(inventory_row["_irwin_norm"])
    incident_name = str(inventory_row["IncidentName"])
    paths = incident_paths(irwin)
    incident_start = time.perf_counter()

    try:
        if RESUME and incident_complete(irwin):
            metadata = json.loads(paths["metadata"].read_text(encoding="utf-8"))
            metadata["status"] = "SKIPPED_COMPLETE"
            run_log_rows.append(metadata)
            outer.set_postfix({"incident": incident_name[:14], "status": "resume"})
            pd.DataFrame(run_log_rows).to_csv(RUN_LOG_PATH, index=False)
            continue

        old_segments = pd.read_parquet(
            Path(inventory_row["segment_path"]),
            columns=OLD_SEGMENT_COLUMNS,
        )
        old_segments["_irwin_norm"] = old_segments["_irwin_norm"].astype(str)
        old_segments["_geom_hash"] = old_segments["_geom_hash"].astype(str)

        # ========================================================
        # A. Corrected unit geometry
        # ========================================================
        stage_start = time.perf_counter()

        if RESUME and paths["units"].exists():
            units = gpd.read_parquet(paths["units"])
            unit_source = "reused"
            unit_meta = {}
        else:
            if not paths["canonical"].exists():
                raise RuntimeError(f"Canonical shard missing: {paths['canonical']}")
            canonical_parents = gpd.read_parquet(paths["canonical"])
            units, unit_meta = build_incident_units(
                old_segments=old_segments,
                canonical_parents=canonical_parents,
            )
            units.to_parquet(paths["units"], index=False)
            unit_source = "new"
            del canonical_parents

        unit_seconds = time.perf_counter() - stage_start

        # ========================================================
        # B. Within-unit sample points
        # ========================================================
        stage_start = time.perf_counter()

        if RESUME and paths["samples"].exists():
            sample_points = pd.read_parquet(paths["samples"])
            sample_source = "reused"
        else:
            sample_points = build_sample_points(units)
            sample_points.to_parquet(paths["samples"], index=False)
            sample_source = "new"

        sample_seconds = time.perf_counter() - stage_start

        if len(sample_points) == 0:
            raise RuntimeError("No within-unit sample points generated.")

        # ========================================================
        # C. NIROPS progression
        # ========================================================
        incident_nirops = nirops_metric[
            nirops_metric["_irwin_norm"] == irwin
        ].copy()
        progression = prepare_progression(incident_nirops)

        if len(progression) < MIN_NIROPS_OBSERVATIONS:
            raise RuntimeError(
                f"Only {len(progression)} NIROPS observations after preparation."
            )

        times = pd.DatetimeIndex(progression["_time"])

        # ========================================================
        # D. Signed sample-point trajectories
        # ========================================================
        stage_start = time.perf_counter()

        if RESUME and paths["trajectory"].exists():
            with np.load(paths["trajectory"], allow_pickle=False) as cached:
                signed = cached["signed"].astype(np.float32, copy=False)
                cached_times = cached["times_ns"]
                raster_mode = str(cached["raster_mode"][0])
                raster_regions = int(cached["raster_regions"][0])

            if not np.array_equal(cached_times, times.asi8):
                raise RuntimeError("Cached trajectory timestamps do not match NIROPS progression.")

            trajectory_source = "reused"
        else:
            signed, region_info = build_incident_trajectory(
                sample_points=sample_points,
                progression=progression,
                incident_name=incident_name,
            )
            raster_mode = region_info["mode"]
            raster_regions = len(region_info["regions"])

            # Uncompressed NPZ is faster to write/read and keeps the
            # expensive raster result available for later 04A features.
            np.savez(
                paths["trajectory"],
                signed=signed.astype(np.float32, copy=False),
                times_ns=times.asi8,
                raster_mode=np.asarray([raster_mode]),
                raster_regions=np.asarray([raster_regions], dtype=np.int32),
            )
            trajectory_source = "new"

        trajectory_seconds = time.perf_counter() - stage_start

        if signed.shape != (len(sample_points), len(progression)):
            raise RuntimeError(
                f"Trajectory shape {signed.shape} does not match "
                f"{len(sample_points)} samples x {len(progression)} times."
            )

        # ========================================================
        # E. Point + unit temporal evidence
        # ========================================================
        stage_start = time.perf_counter()

        if (
            RESUME
            and paths["point_evidence"].exists()
            and paths["unit_evidence"].exists()
        ):
            point_table = pd.read_parquet(paths["point_evidence"])
            unit_evidence = pd.read_parquet(paths["unit_evidence"])
            evidence_source = "reused"
        else:
            point_evidence = classify_sample_points(
                labels=sample_points["_engagement_norm"].to_numpy(dtype=object),
                signed=signed,
                times=times,
            )
            unit_evidence, point_table = aggregate_unit_evidence(
                units=units,
                sample_points=sample_points,
                point_evidence=point_evidence,
                times=times,
            )
            point_table.to_parquet(paths["point_evidence"], index=False)
            unit_evidence.to_parquet(paths["unit_evidence"], index=False)
            evidence_source = "new"

        evidence_seconds = time.perf_counter() - stage_start

        supported_units = int(
            (unit_evidence["unit_evidence"] == "temporally_supported").sum()
        )
        conflict_units = int(
            (unit_evidence["unit_evidence"] == "trajectory_conflict").sum()
        )
        insufficient_units = int(
            (unit_evidence["unit_evidence"] == "insufficient_temporal_evidence").sum()
        )

        metadata = {
            "_irwin_norm": irwin,
            "IncidentName": incident_name,
            "Year": str(inventory_row["Year"]),
            "status": "OK",
            "old_03g_units": int(len(old_segments)),
            "corrected_units": int(len(units)),
            "unit_reduction": int(len(old_segments) - len(units)),
            "unit_reduction_pct": float(
                (len(old_segments) - len(units)) / len(old_segments) * 100.0
            ),
            "sample_points": int(len(sample_points)),
            "mean_sample_points_per_unit": float(len(sample_points) / len(units)),
            "nirops_observations": int(len(progression)),
            "raster_mode": raster_mode,
            "raster_regions": int(raster_regions),
            "supported_units": supported_units,
            "conflict_units": conflict_units,
            "insufficient_units": insufficient_units,
            "unit_source": unit_source,
            "sample_source": sample_source,
            "trajectory_source": trajectory_source,
            "evidence_source": evidence_source,
            "unit_build_seconds": float(unit_seconds),
            "sample_seconds": float(sample_seconds),
            "trajectory_seconds": float(trajectory_seconds),
            "evidence_seconds": float(evidence_seconds),
            "runtime_seconds": float(time.perf_counter() - incident_start),
            **unit_meta,
        }

        paths["metadata"].write_text(
            json.dumps(metadata, indent=2, default=str),
            encoding="utf-8",
        )

        if paths["error"].exists():
            paths["error"].unlink()

        run_log_rows.append(metadata)
        outer.set_postfix({
            "incident": incident_name[:13],
            "units": len(units),
            "pts": len(sample_points),
            "raster": raster_mode,
            "sec": f"{metadata['runtime_seconds']:.1f}",
        })

        del (
            old_segments, units, sample_points, progression,
            signed, point_table, unit_evidence,
        )
        gc.collect()

    except Exception as exc:
        error_text = traceback.format_exc()
        paths["error"].write_text(error_text, encoding="utf-8")
        row = {
            "_irwin_norm": irwin,
            "IncidentName": incident_name,
            "Year": str(inventory_row["Year"]),
            "status": "ERROR",
            "error": f"{type(exc).__name__}: {exc}",
            "runtime_seconds": float(time.perf_counter() - incident_start),
        }
        run_log_rows.append(row)
        outer.set_postfix({"incident": incident_name[:13], "status": "ERROR"})

        # Release any large partial arrays before the next incident.
        signed = None
        units = None
        sample_points = None
        point_table = None
        unit_evidence = None
        gc.collect()

    pd.DataFrame(run_log_rows).to_csv(RUN_LOG_PATH, index=False)


## Coverage and combined corrected evidence

In [ ]:

metadata_rows = []
for path_meta in sorted(INCIDENT_META_ROOT.glob("*.json")):
    try:
        metadata_rows.append(json.loads(path_meta.read_text(encoding="utf-8")))
    except Exception:
        pass

production_metadata = pd.DataFrame(metadata_rows)

if len(production_metadata):
    completed_ids = set(
        production_metadata.loc[
            production_metadata["status"] == "OK",
            "_irwin_norm",
        ].astype(str)
    )
else:
    completed_ids = set()

missing_ids = [x for x in PRODUCTION_IDS if x not in completed_ids]

coverage_summary = pd.DataFrame([{
    "target_incidents": len(PRODUCTION_IDS),
    "completed_incidents": len(completed_ids),
    "missing_or_failed_incidents": len(missing_ids),
    "completion_pct": len(completed_ids) / len(PRODUCTION_IDS) * 100.0,
    "old_03g_units_completed": int(production_metadata["old_03g_units"].sum()) if len(production_metadata) else 0,
    "corrected_units_completed": int(production_metadata["corrected_units"].sum()) if len(production_metadata) else 0,
    "sample_points_completed": int(production_metadata["sample_points"].sum()) if len(production_metadata) else 0,
    "supported_units_completed": int(production_metadata["supported_units"].sum()) if len(production_metadata) else 0,
    "single_grid_incidents": int((production_metadata["raster_mode"] == "single").sum()) if len(production_metadata) else 0,
    "tiled_incidents": int((production_metadata["raster_mode"] == "tiled").sum()) if len(production_metadata) else 0,
}])

display(coverage_summary)
coverage_summary.to_csv(OUTPUT_ROOT / "production_coverage_summary.csv", index=False)

if missing_ids:
    pd.DataFrame({"_irwin_norm": missing_ids}).to_csv(
        OUTPUT_ROOT / "missing_or_failed_incidents.csv",
        index=False,
    )

frames = []
for irwin in tqdm(sorted(completed_ids), desc="Combine corrected unit evidence"):
    p = UNIT_EVIDENCE_ROOT / f"{irwin}.parquet"
    if p.exists():
        frames.append(pd.read_parquet(p))

corrected_evidence = (
    pd.concat(frames, ignore_index=True)
    if frames
    else pd.DataFrame()
)
del frames

if len(corrected_evidence):
    assert corrected_evidence["unit_uid"].is_unique

print("Combined corrected units:", f"{len(corrected_evidence):,}")


## Corrected evidence, bracket sensitivity, and candidate supported cohort

In [ ]:

if len(corrected_evidence):
    corrected_evidence_summary = (
        corrected_evidence.groupby(
            ["_engagement_norm", "unit_evidence", "unit_diagnostic_reason"],
            dropna=False,
        )
        .agg(
            units=("unit_uid", "size"),
            incidents=("_irwin_norm", "nunique"),
            line_length_km=(
                "actual_line_length_m",
                lambda s: s.sum() / 1000.0,
            ),
        )
        .reset_index()
    )
    corrected_evidence_summary["pct_within_label"] = (
        corrected_evidence_summary["units"]
        / corrected_evidence_summary.groupby("_engagement_norm")["units"].transform("sum")
        * 100.0
    )

    bracket_rows = []
    for label, group in corrected_evidence.groupby("_engagement_norm"):
        bracket_rows.append({
            "_engagement_norm": label,
            "units": len(group),
            "any_supported": int(group["any_supported"].sum()),
            "any_supported_pct": group["any_supported"].mean() * 100.0,
            "majority_supported": int(group["majority_supported"].sum()),
            "majority_supported_pct": group["majority_supported"].mean() * 100.0,
            "all_supported": int(group["all_supported"].sum()),
            "all_supported_pct": group["all_supported"].mean() * 100.0,
            "primary_unit_rule_supported": int(
                (group["unit_evidence"] == "temporally_supported").sum()
            ),
            "primary_unit_rule_supported_pct": (
                (group["unit_evidence"] == "temporally_supported").mean() * 100.0
            ),
        })
    unit_support_brackets = pd.DataFrame(bracket_rows)

    corrected_supported = corrected_evidence[
        corrected_evidence["unit_evidence"] == "temporally_supported"
    ].copy()
    corrected_supported["hold_label"] = (
        corrected_supported["_engagement_norm"] == "Held"
    ).astype(np.int8)

    CORRECTED_COHORT_PATH = (
        PROCESSED_ROOT / "corrected_supported_cohort_candidate.parquet"
    )
    corrected_supported.to_parquet(CORRECTED_COHORT_PATH, index=False)
else:
    corrected_evidence_summary = pd.DataFrame()
    unit_support_brackets = pd.DataFrame()
    corrected_supported = pd.DataFrame()
    CORRECTED_COHORT_PATH = None

display(corrected_evidence_summary)
display(unit_support_brackets)

corrected_evidence_summary.to_csv(
    OUTPUT_ROOT / "corrected_unit_evidence_summary.csv",
    index=False,
)
unit_support_brackets.to_csv(
    OUTPUT_ROOT / "unit_support_bracket_sensitivity.csv",
    index=False,
)

print("Candidate corrected supported cohort:", f"{len(corrected_supported):,}")
if len(corrected_supported):
    print("Supported incidents:", corrected_supported["_irwin_norm"].nunique())
    print("Held:", f"{(corrected_supported['_engagement_norm'] == 'Held').sum():,}")
    print("Burned Over:", f"{(corrected_supported['_engagement_norm'] == 'Burned Over').sum():,}")


## Incident concentration and comparison with 03H_v2

In [ ]:

def concentration_metrics(frame, label):
    counts = (
        frame.groupby("_irwin_norm")
        .size()
        .sort_values(ascending=False)
        .to_numpy(dtype=float)
    )
    if not len(counts):
        return {"group": label}
    total = counts.sum()
    def top_pct(n):
        return counts[:min(n, len(counts))].sum() / total * 100.0
    return {
        "group": label,
        "incidents": len(counts),
        "units": int(total),
        "top1_pct": top_pct(1),
        "top5_pct": top_pct(5),
        "top10_pct": top_pct(10),
        "top20_pct": top_pct(20),
        "effective_incidents": total**2 / np.square(counts).sum(),
    }

concentration_rows = []
if len(corrected_supported):
    concentration_rows.append(
        concentration_metrics(corrected_supported, "All corrected supported")
    )
    for label in ENGAGED_LABELS:
        concentration_rows.append(
            concentration_metrics(
                corrected_supported[
                    corrected_supported["_engagement_norm"] == label
                ],
                label,
            )
        )

corrected_concentration = pd.DataFrame(concentration_rows)
display(corrected_concentration)
corrected_concentration.to_csv(
    OUTPUT_ROOT / "corrected_cohort_concentration.csv",
    index=False,
)

old_rows = np.nan
old_incidents = np.nan
if OLD_ML_COHORT_PATH.exists():
    old = pd.read_parquet(
        OLD_ML_COHORT_PATH,
        columns=["_irwin_norm", "segment_uid"],
    )
    old_rows = len(old)
    old_incidents = old["_irwin_norm"].nunique()
    del old

old_vs_corrected = pd.DataFrame([
    {
        "metric": "supported modeling units",
        "03H_v2_midpoint": old_rows,
        "03L_corrected_multipoint_candidate": len(corrected_supported),
    },
    {
        "metric": "supported incidents",
        "03H_v2_midpoint": old_incidents,
        "03L_corrected_multipoint_candidate": (
            corrected_supported["_irwin_norm"].nunique()
            if len(corrected_supported)
            else 0
        ),
    },
])
display(old_vs_corrected)
old_vs_corrected.to_csv(
    OUTPUT_ROOT / "old_vs_corrected_high_level_comparison.csv",
    index=False,
)


## Runtime summary

In [ ]:

if len(production_metadata):
    runtime_summary = pd.DataFrame([{
        "completed_incidents": len(production_metadata),
        "sum_incident_runtime_hours": production_metadata["runtime_seconds"].sum() / 3600.0,
        "median_incident_seconds": production_metadata["runtime_seconds"].median(),
        "p90_incident_seconds": production_metadata["runtime_seconds"].quantile(0.90),
        "max_incident_seconds": production_metadata["runtime_seconds"].max(),
        "unit_build_minutes": production_metadata["unit_build_seconds"].sum() / 60.0,
        "sampling_minutes": production_metadata["sample_seconds"].sum() / 60.0,
        "trajectory_minutes": production_metadata["trajectory_seconds"].sum() / 60.0,
        "evidence_minutes": production_metadata["evidence_seconds"].sum() / 60.0,
    }])
else:
    runtime_summary = pd.DataFrame()

display(runtime_summary)
runtime_summary.to_csv(OUTPUT_ROOT / "runtime_summary.csv", index=False)


## Production report

In [ ]:

def df_to_md(df, max_rows=200):
    if df is None or len(df) == 0:
        return "_No rows._"
    try:
        return df.head(max_rows).to_markdown(index=False)
    except Exception:
        return df.head(max_rows).to_string(index=False)

report = f"""# Corrected Local Fireline Unit Production

Notebook build: `{NOTEBOOK_BUILD_ID}`

Generated: {datetime.now(timezone.utc).isoformat()}

## Fixed geometry

Maximum actual line length per unit: **{SEGMENT_LENGTH_M:g} m**

Maximum short-fragment endpoint gap: **{FRAGMENT_GAP_M:g} m**

Within-unit sampling: **approximately {UNIT_SAMPLE_SPACING_M:g} m**, with at least one sample per constituent geometry part.

## Fixed temporal reconstruction

Raster: **{RASTER_RESOLUTION_M:g} m**

Challenge tolerance: **{PRIMARY_TOLERANCE_M:g} m**

Maximum NIROPS interval: **{PRIMARY_MAX_GAP_HOURS:g} h**

## Coverage

{df_to_md(coverage_summary)}

## Corrected unit evidence

{df_to_md(corrected_evidence_summary)}

## Any / majority / all-supported sensitivity

{df_to_md(unit_support_brackets)}

## Corrected supported-cohort concentration

{df_to_md(corrected_concentration)}

## 03H_v2 versus corrected representation

{df_to_md(old_vs_corrected)}

## Runtime

{df_to_md(runtime_summary)}

## Candidate unit evidence rule

`{UNIT_EVIDENCE_RULE}`

The rule is intentionally asymmetric because the event semantics differ:

- an observed crossing at any sampled location corroborates a Burned Over local unit;
- a Held local unit requires observed approach and no sampled Held trajectory conflict.

The any/majority/all-supported columns are retained so this rule can be reviewed without rerunning NIROPS.

## Speed / reuse

Canonical parent geometry is sharded once.

Per-incident corrected unit geometry, sample points, signed NIROPS trajectories, point evidence, and unit evidence are separately cached.

The signed-distance NPZ files should be reused directly by the later dynamic-approach feature notebook rather than rerasterizing NIROPS.

## Candidate corrected supported cohort

`{CORRECTED_COHORT_PATH}`

Do not call the corrected cohort final until the primary unit-evidence rule and the concentration diagnostics have been reviewed.
"""

REPORT_PATH = OUTPUT_ROOT / "CORRECTED_LOCAL_UNIT_PRODUCTION_REPORT.md"
REPORT_PATH.write_text(report, encoding="utf-8")

print(report)
print("\nSaved:", REPORT_PATH)


## Output inventory

In [ ]:

print("=" * 84)
print("03L OUTPUTS")
print("=" * 84)

for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        size = (
            f"{p.stat().st_size / (1024**2):,.2f} MB"
            if p.stat().st_size >= 1024**2
            else f"{p.stat().st_size / 1024:,.1f} KB"
        )
        print(f"{str(p.relative_to(OUTPUT_ROOT)):<72}{size:>15}")

print()
for directory in [
    CANONICAL_SHARD_ROOT, UNIT_ROOT, SAMPLE_ROOT, TRAJECTORY_ROOT,
    POINT_EVIDENCE_ROOT, UNIT_EVIDENCE_ROOT, INCIDENT_META_ROOT,
]:
    print(f"{directory.name:<30}{len(list(directory.glob('*'))):>8} files")

print("\nDONE:", NOTEBOOK_BUILD_ID)
